# Synthetic ECG validation: Healthy male vs female (beat-level aggregation)

This notebook reproduces a paper-style qualitative plot using the balanced-sex SSSD-ECG generator (`20000.pkl`):

- Condition on **NORM only**.
- Generate **50 male** and **50 female** synthetic ECGs.
- Detect R-peaks, crop beats with fixed window (**300 ms pre**, **500 ms post**).
- Aggregate beats on **lead V1** using median and IQR (25th–75th percentile).
- Plot male (blue) vs female (red).

In [ ]:
!pip install -q pykeops wfdb resampy ishneholterlib pytorch-lightning neurokit2

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import neurokit2 as nk
from scipy.signal import find_peaks

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path('/content/CMED-Mini-Project')
REPO_ROOT = PROJECT_ROOT / 'SSSD-ECG'

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'src' / 'ptb_xl'))
sys.path.insert(0, str(REPO_ROOT / 'src' / 'sssd'))

from models.SSSD_ECG import SSSD_ECG
from utils.util import calc_diffusion_hyperparams, sampling_label
from inference import generate_four_leads
from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

print('REPO_ROOT:', REPO_ROOT)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
from pathlib import Path
import os
from google.colab import drive

zip_src = "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip"
zip_dst = "/content/ptbxl.zip"
raw_dir = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")

if not os.path.exists(zip_src):
    raise FileNotFoundError(f"Missing zip on Drive: {zip_src}")

if not os.path.exists(zip_dst):
    !cp "{zip_src}" "{zip_dst}"

if not raw_dir.exists():
    !unzip -oq "{zip_dst}" -d /content/
    print("PTB-XL extracted.")
else:
    print("PTB-XL already extracted.")


In [ ]:
# Load balanced-sex generator config + strict checkpoint=20000
%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd

import json
from pathlib import Path
import numpy as np
import torch

from models.SSSD_ECG import SSSD_ECG
from utils.util import calc_diffusion_hyperparams
from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

config_dir = Path("config")
config_dir.mkdir(parents=True, exist_ok=True)
cfg_path = config_dir / "config_SSSD_ECG_sex_scp71_balanced_50f_50m.json"

# If missing (new session), recreate with the same settings used in training
if not cfg_path.exists():
    drive_output_dir = "/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m"
    default_cfg = {
        "diffusion_config": {
            "T": 200,
            "beta_0": 0.0001,
            "beta_T": 0.02
        },
        "wavenet_config": {
            "in_channels": 8,
            "out_channels": 8,
            "num_res_layers": 36,
            "res_channels": 256,
            "skip_channels": 256,
            "diffusion_step_embed_dim_in": 128,
            "diffusion_step_embed_dim_mid": 512,
            "diffusion_step_embed_dim_out": 512,
            "s4_lmax": 1000,
            "s4_d_state": 64,
            "s4_dropout": 0.0,
            "s4_bidirectional": 1,
            "s4_layernorm": 1,
            "label_embed_dim": 128,
            "label_embed_classes": 72
        },
        "train_config": {
            "output_directory": drive_output_dir,
            "ckpt_iter": "max",
            "iters_per_ckpt": 2000,
            "iters_per_logging": 1000,
            "n_iters": 21000,
            "learning_rate": 2e-4,
            "batch_size": 8
        },
        "trainset_config": {
            "segment_length": 1000,
            "sampling_rate": 100,
            "finetune_dataset": "ptbxl_sex_scp71_balanced_50f_50m"
        },
        "gen_config": {
            "output_directory": drive_output_dir,
            "ckpt_path": drive_output_dir
        }
    }
    cfg_path.write_text(json.dumps(default_cfg, indent=2))
    print("Recreated missing config:", cfg_path)

cfg = json.loads(cfg_path.read_text())
diffusion_config = cfg["diffusion_config"]
model_config = cfg["wavenet_config"]
train_config = cfg["train_config"]

ckpt_dir = Path(train_config["output_directory"]) / f"ch{model_config['res_channels']}_T{diffusion_config['T']}_betaT{diffusion_config['beta_T']}"
ckpt_path = ckpt_dir / "20000.pkl"

if not ckpt_path.exists():
    raise FileNotFoundError(f"Required checkpoint missing: {ckpt_path}")

# Build label_names so we can locate NORM index in 71-label block
target_fs = 100
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

if target_folder_ptb_xl.exists():
    !rm -rf {target_folder_ptb_xl}

_ = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)
reformat_as_memmap(
    _[0],
    target_folder_ptb_xl / "memmap.npy",
    data_folder=target_folder_ptb_xl,
    delete_npys=True,
)
_, lbl_itos_dict, _, _ = load_dataset(target_folder_ptb_xl)

label_names = np.array(lbl_itos_dict["label_all"])
assert len(label_names) == 71, len(label_names)

if "NORM" not in set(label_names.tolist()):
    raise ValueError("NORM not found in label_all names")
norm_idx = int(np.where(label_names == "NORM")[0][0])

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = SSSD_ECG(**model_config).to(device)
checkpoint = torch.load(ckpt_path, map_location="cpu")
net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

diffusion_hyperparams = calc_diffusion_hyperparams(**diffusion_config)
for k in diffusion_hyperparams:
    if k != "T":
        diffusion_hyperparams[k] = diffusion_hyperparams[k].to(device)

print("Loaded checkpoint:", ckpt_path)
print("NORM index:", norm_idx)

In [ ]:
# Generate healthy (NORM+SR) male/female groups
N_PER_GROUP = 200
FS = 100
LEAD_NAMES = ['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'III', 'aVR', 'aVL', 'aVF']
LEAD_NAME = 'V1'
LEAD_IDX = LEAD_NAMES.index(LEAD_NAME)

if 'SR' not in set(label_names.tolist()):
    raise ValueError('SR not found in label_names')
sr_idx = int(np.where(label_names == 'SR')[0][0])


def make_norm_sr_cond_batch(n: int, sex_binary: float):
    # cond layout: [sex, 71-label-multihot]
    cond = np.zeros((n, 72), dtype=np.float32)
    cond[:, 0] = sex_binary
    cond[:, 1 + norm_idx] = 1.0
    cond[:, 1 + sr_idx] = 1.0
    return cond


def generate_group(n: int, sex_binary: float, seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    cond_np = make_norm_sr_cond_batch(n=n, sex_binary=sex_binary)
    cond = torch.tensor(cond_np, dtype=torch.float32, device=device)
    with torch.no_grad():
        gen8 = sampling_label(net, (n, 8, 1000), diffusion_hyperparams, cond=cond)
        gen12 = generate_four_leads(gen8)
    return gen12.detach().cpu().numpy().astype(np.float32), cond_np

male_ecg, male_cond = generate_group(N_PER_GROUP, sex_binary=0.0, seed=2026)
female_ecg, female_cond = generate_group(N_PER_GROUP, sex_binary=1.0, seed=3036)

print('male_ecg:', male_ecg.shape, 'female_ecg:', female_ecg.shape)
print('male cond unique sex:', np.unique(male_cond[:, 0]), 'female cond unique sex:', np.unique(female_cond[:, 0]))
print('Condition labels active: NORM and SR')
print('Lead selected:', LEAD_NAME, '(index', LEAD_IDX, ')')



In [ ]:
# Beat-level aggregation: R-peak detect -> crop [-300ms, +500ms] -> median/IQR
PRE_MS = 300
POST_MS = 500
PRE = int(round(FS * PRE_MS / 1000.0))   # 30 samples
POST = int(round(FS * POST_MS / 1000.0)) # 50 samples
WIN = PRE + POST


def detect_rpeaks_1d(sig_1d: np.ndarray, fs: int = 100):
    # Try NeuroKit first, fallback to scipy if needed
    try:
        _, info = nk.ecg_peaks(sig_1d, sampling_rate=fs)
        peaks = np.asarray(info.get('ECG_R_Peaks', []), dtype=int)
        if len(peaks) > 0:
            return peaks
    except Exception:
        pass

    # fallback
    prominence = max(0.02, float(np.std(sig_1d)) * 0.5)
    peaks, _ = find_peaks(sig_1d, distance=int(0.25 * fs), prominence=prominence)
    return peaks.astype(int)


def collect_beats(ecg_12: np.ndarray, lead_idx: int, fs: int = 100):
    # ecg_12 shape: (N, 12, 1000)
    beats = []
    n_total_peaks = 0
    for i in range(ecg_12.shape[0]):
        lead = ecg_12[i, lead_idx]
        peaks = detect_rpeaks_1d(lead, fs=fs)
        n_total_peaks += len(peaks)
        for p in peaks:
            s = p - PRE
            e = p + POST
            if s >= 0 and e <= lead.shape[0]:
                beats.append(lead[s:e])
    if len(beats) == 0:
        raise RuntimeError('No valid cropped beats found. Try adjusting detection params.')
    beats = np.stack(beats).astype(np.float32)  # (n_beats, WIN)
    return beats, n_total_peaks


male_beats, male_r_all = collect_beats(male_ecg, LEAD_IDX, fs=FS)
female_beats, female_r_all = collect_beats(female_ecg, LEAD_IDX, fs=FS)

print('male beats:', male_beats.shape, '| detected peaks total:', male_r_all)
print('female beats:', female_beats.shape, '| detected peaks total:', female_r_all)


def agg_curves(beats: np.ndarray):
    med = np.median(beats, axis=0)
    p25 = np.percentile(beats, 25, axis=0)
    p75 = np.percentile(beats, 75, axis=0)
    return med, p25, p75


male_med, male_p25, male_p75 = agg_curves(male_beats)
female_med, female_p25, female_p75 = agg_curves(female_beats)

In [ ]:
# Plot male vs female (paper-style) for lead V1
t_ms = (np.arange(WIN) - PRE) * (1000.0 / FS)
FS_PLOT = 20

plt.figure(figsize=(10, 5))

# Male (blue)
plt.plot(t_ms, male_med, color='royalblue', linewidth=1.0, label='Male')
plt.fill_between(t_ms, male_p25, male_p75, color='royalblue', alpha=0.20)

# Female (red)
plt.plot(t_ms, female_med, color='crimson', linewidth=1.0, label='Female')
plt.fill_between(t_ms, female_p25, female_p75, color='crimson', alpha=0.20)

plt.axvline(0, color='k', linestyle='--', linewidth=1, alpha=0.6)
plt.xlabel('Time relative to R-peak (ms)', fontsize=FS_PLOT)
plt.ylabel('mV', fontsize=FS_PLOT)
plt.xticks(fontsize=FS_PLOT)
plt.yticks(fontsize=FS_PLOT)
plt.legend(loc='best', frameon=False, fontsize=FS_PLOT)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Additional separate healthy male vs female plots for V2 and V5
FS_PLOT = 20
for lead_name in ['V2', 'V5']:
    lead_idx = LEAD_NAMES.index(lead_name)

    male_beats_l, _ = collect_beats(male_ecg, lead_idx, fs=FS)
    female_beats_l, _ = collect_beats(female_ecg, lead_idx, fs=FS)

    male_med_l, male_p25_l, male_p75_l = agg_curves(male_beats_l)
    female_med_l, female_p25_l, female_p75_l = agg_curves(female_beats_l)

    t_ms = (np.arange(WIN) - PRE) * (1000.0 / FS)

    plt.figure(figsize=(10, 5))
    plt.plot(t_ms, male_med_l, color='royalblue', linewidth=1.0, label='Male')
    plt.fill_between(t_ms, male_p25_l, male_p75_l, color='royalblue', alpha=0.20)

    plt.plot(t_ms, female_med_l, color='crimson', linewidth=1.0, label='Female')
    plt.fill_between(t_ms, female_p25_l, female_p75_l, color='crimson', alpha=0.20)

    plt.axvline(0, color='k', linestyle='--', linewidth=1, alpha=0.6)
    plt.xlabel('Time relative to R-peak (ms)', fontsize=FS_PLOT)
    plt.ylabel('mV', fontsize=FS_PLOT)
    plt.xticks(fontsize=FS_PLOT)
    plt.yticks(fontsize=FS_PLOT)
    plt.legend(loc='best', frameon=False, fontsize=FS_PLOT)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

## Synthetic validation: Male healthy (NORM) vs male MI

This section automatically picks the most frequent MI statement in the **male train split**, optionally adds a strongly co-occurring rhythm label (if present and common), then compares beat-level aggregated morphology against male NORM-only generation.

In [ ]:
# Fixed male MI condition: ABQRS + IMI + SR
# Assumes label_names, target_folder_ptb_xl, norm_idx are already defined.

N_PER_GROUP = 100

required_labels = ['ABQRS', 'IMI', 'SR']
label_to_idx = {str(n): i for i, n in enumerate(label_names)}
missing = [x for x in required_labels if x not in label_to_idx]
if len(missing) > 0:
    raise ValueError(f'Missing required labels in label_names: {missing}')

abqrs_idx = int(label_to_idx['ABQRS'])
imi_idx = int(label_to_idx['IMI'])
sr_idx = int(label_to_idx['SR'])


def make_cond_batch(n: int, sex_binary: float, active_label_indices):
    cond = np.zeros((n, 72), dtype=np.float32)
    cond[:, 0] = sex_binary
    for idx in active_label_indices:
        cond[:, 1 + int(idx)] = 1.0
    return cond

# Male healthy condition (NORM only)
male_norm_cond = make_cond_batch(N_PER_GROUP, sex_binary=0.0, active_label_indices=[norm_idx])

# Male MI condition (fixed): ABQRS + IMI + SR
mi_active = [abqrs_idx, imi_idx, sr_idx]
male_mi_cond = make_cond_batch(N_PER_GROUP, sex_binary=0.0, active_label_indices=mi_active)

# Generate groups
with torch.no_grad():
    cond = torch.tensor(male_norm_cond, dtype=torch.float32, device=device)
    gen8 = sampling_label(net, (N_PER_GROUP, 8, 1000), diffusion_hyperparams, cond=cond)
    male_norm_ecg = generate_four_leads(gen8).detach().cpu().numpy().astype(np.float32)

with torch.no_grad():
    cond = torch.tensor(male_mi_cond, dtype=torch.float32, device=device)
    gen8 = sampling_label(net, (N_PER_GROUP, 8, 1000), diffusion_hyperparams, cond=cond)
    male_mi_ecg = generate_four_leads(gen8).detach().cpu().numpy().astype(np.float32)

mi_lbl = 'ABQRS + IMI + SR'
print('male_norm_ecg:', male_norm_ecg.shape)
print('male_mi_ecg:', male_mi_ecg.shape)
print('MI condition active labels:', [str(label_names[i]) for i in mi_active])



In [ ]:
# Beat-level aggregation + plot: male healthy vs male MI for V1, V2, V3, V5
# Same paper-style format as before (thin lines, no title, simple legend, larger fonts)

FS_PLOT = 20
LEADS_TO_PLOT = ['V1', 'V2', 'V3', 'V5']

# If mi_lbl is not already defined in the generation cell, fallback:
if 'mi_lbl' not in globals():
    mi_lbl = 'ABQRS + IMI + SR'

for lead_name in LEADS_TO_PLOT:
    lead_idx = LEAD_NAMES.index(lead_name)

    male_norm_beats, male_norm_r_all = collect_beats(male_norm_ecg, lead_idx, fs=FS)
    male_mi_beats, male_mi_r_all = collect_beats(male_mi_ecg, lead_idx, fs=FS)

    male_norm_med, male_norm_p25, male_norm_p75 = agg_curves(male_norm_beats)
    male_mi_med, male_mi_p25, male_mi_p75 = agg_curves(male_mi_beats)

    print(f'[{lead_name}] male_norm beats:', male_norm_beats.shape, '| detected peaks total:', male_norm_r_all)
    print(f'[{lead_name}] male_mi beats:', male_mi_beats.shape, '| detected peaks total:', male_mi_r_all)

    t_ms = (np.arange(WIN) - PRE) * (1000.0 / FS)

    plt.figure(figsize=(10, 5))

    # Healthy male (blue)
    plt.plot(t_ms, male_norm_med, color='royalblue', linewidth=1.0, label='Male healthy')
    plt.fill_between(t_ms, male_norm_p25, male_norm_p75, color='royalblue', alpha=0.20)

    # MI male (red)
    plt.plot(t_ms, male_mi_med, color='crimson', linewidth=1.0, label='Male MI')
    plt.fill_between(t_ms, male_mi_p25, male_mi_p75, color='crimson', alpha=0.20)

    plt.axvline(0, color='k', linestyle='--', linewidth=1, alpha=0.6)
    plt.xlabel('Time relative to R-peak (ms)', fontsize=FS_PLOT)
    plt.ylabel('mV', fontsize=FS_PLOT)
    plt.xticks(fontsize=FS_PLOT)
    plt.yticks(fontsize=FS_PLOT)
    plt.legend(loc='best', frameon=False, fontsize=FS_PLOT)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

## Automated statistical tests (synthetic and real, male vs female, strict NORM)

Definitions used:
- **ST@J**: amplitude at the J-point (QRS offset).
- **ST@J+40ms**: amplitude 40 ms after the J-point.

This section computes beat-level features with NeuroKit2 delineation and runs Mann-Whitney U tests with PASS/FAIL checks for expected clinical direction.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor
from clinical_ts.timeseries_utils import load_dataset
import pickle

# ------------------------------
# Helpers
# ------------------------------

def _to_idx(arr, j):
    if arr is None or j >= len(arr):
        return None
    v = arr[j]
    if v is None:
        return None
    try:
        if np.isnan(v):
            return None
    except Exception:
        pass
    return int(v)


def extract_features_from_group(ecg_12_batch, fs=100):
    """
    Beat-level pooled features from a group of ECGs.
    - QRS/QTc from lead II
    - ST and T-wave features kept SEPARATE per lead (V2, V5)
    """
    lead_ii = LEAD_NAMES.index('II')
    lead_v2 = LEAD_NAMES.index('V2')
    lead_v5 = LEAD_NAMES.index('V5')

    qrs_ms = []
    qtc_ms = []

    st_j_v2 = []
    st_j40_v2 = []
    st_j_v5 = []
    st_j40_v5 = []

    t_peak_v2 = []
    t_peak_v5 = []

    j40 = int(round(0.040 * fs))

    for i in range(ecg_12_batch.shape[0]):
        sig_ii = ecg_12_batch[i, lead_ii]
        sig_v2 = ecg_12_batch[i, lead_v2]
        sig_v5 = ecg_12_batch[i, lead_v5]

        # R-peaks on lead II
        rpeaks = detect_rpeaks_1d(sig_ii, fs=fs)
        if len(rpeaks) < 2:
            continue

        try:
            _, waves_ii = nk.ecg_delineate(sig_ii, rpeaks, sampling_rate=fs, method='dwt', show=False)
            _, waves_v2 = nk.ecg_delineate(sig_v2, rpeaks, sampling_rate=fs, method='dwt', show=False)
            _, waves_v5 = nk.ecg_delineate(sig_v5, rpeaks, sampling_rate=fs, method='dwt', show=False)
        except Exception:
            continue

        on_ii = waves_ii.get('ECG_R_Onsets', None)
        off_ii = waves_ii.get('ECG_R_Offsets', None)
        toff_ii = waves_ii.get('ECG_T_Offsets', None)

        off_v2 = waves_v2.get('ECG_R_Offsets', None)
        off_v5 = waves_v5.get('ECG_R_Offsets', None)
        tpk_v2 = waves_v2.get('ECG_T_Peaks', None)
        tpk_v5 = waves_v5.get('ECG_T_Peaks', None)

        nbeats = len(rpeaks)
        for j in range(nbeats):
            # --- QRS and QTc (lead II) ---
            q_on = _to_idx(on_ii, j)
            q_off = _to_idx(off_ii, j)
            t_off = _to_idx(toff_ii, j)

            if q_on is not None and q_off is not None and q_off > q_on:
                qrs_ms.append((q_off - q_on) * 1000.0 / fs)

            rr = None
            if j < nbeats - 1:
                rr = (rpeaks[j + 1] - rpeaks[j]) / fs
            elif j > 0:
                rr = (rpeaks[j] - rpeaks[j - 1]) / fs

            if (q_on is not None) and (t_off is not None) and (t_off > q_on) and (rr is not None) and (rr > 1e-6):
                qt_sec = (t_off - q_on) / fs
                qtc_ms.append((qt_sec / np.sqrt(rr)) * 1000.0)

            # --- ST features, per lead (do NOT average V2 with V5) ---
            j_v2 = _to_idx(off_v2, j)
            if j_v2 is not None and (j_v2 + j40) < len(sig_v2):
                st_j_v2.append(float(sig_v2[j_v2]))
                st_j40_v2.append(float(sig_v2[j_v2 + j40]))

            j_v5 = _to_idx(off_v5, j)
            if j_v5 is not None and (j_v5 + j40) < len(sig_v5):
                st_j_v5.append(float(sig_v5[j_v5]))
                st_j40_v5.append(float(sig_v5[j_v5 + j40]))

            # --- T-wave peak amplitude, per lead ---
            t_v2 = _to_idx(tpk_v2, j)
            if t_v2 is not None and 0 <= t_v2 < len(sig_v2):
                t_peak_v2.append(float(sig_v2[t_v2]))

            t_v5 = _to_idx(tpk_v5, j)
            if t_v5 is not None and 0 <= t_v5 < len(sig_v5):
                t_peak_v5.append(float(sig_v5[t_v5]))

    return {
        'QRS Duration (ms)': np.asarray(qrs_ms, dtype=np.float64),
        'QTc (ms)': np.asarray(qtc_ms, dtype=np.float64),
        'ST@J V2 (mV)': np.asarray(st_j_v2, dtype=np.float64),
        'ST@J+40ms V2 (mV)': np.asarray(st_j40_v2, dtype=np.float64),
        'ST@J V5 (mV)': np.asarray(st_j_v5, dtype=np.float64),
        'ST@J+40ms V5 (mV)': np.asarray(st_j40_v5, dtype=np.float64),
        'T-wave Peak V2 (mV)': np.asarray(t_peak_v2, dtype=np.float64),
        'T-wave Peak V5 (mV)': np.asarray(t_peak_v5, dtype=np.float64),
    }


def run_tests(source_name, male_feats, female_feats):
    # Direction assumptions kept explicit for PASS/FAIL checks.
    specs = [
        ('QRS Duration (ms)', 'male > female', lambda m, f: m > f),
        ('QTc (ms)', 'female > male', lambda m, f: f > m),
        ('ST@J V2 (mV)', 'male > female', lambda m, f: m > f),
        ('ST@J+40ms V2 (mV)', 'male > female', lambda m, f: m > f),
        ('ST@J V5 (mV)', 'male > female', lambda m, f: m > f),
        ('ST@J+40ms V5 (mV)', 'male > female', lambda m, f: m > f),
        ('T-wave Peak V2 (mV)', 'male > female', lambda m, f: m > f),
        ('T-wave Peak V5 (mV)', 'male > female', lambda m, f: m > f),
    ]

    rows = []
    for feat, expected_dir, dir_ok_fn in specs:
        m = male_feats[feat]
        f = female_feats[feat]

        if len(m) == 0 or len(f) == 0:
            rows.append({
                'Source': source_name,
                'Feature': feat,
                'Male Median': np.nan,
                'Female Median': np.nan,
                'Abs Diff': np.nan,
                'Expected Direction': expected_dir,
                'Observed Direction': 'insufficient data',
                'p_value': np.nan,
                'PASS': False,
                'N_male_beats': int(len(m)),
                'N_female_beats': int(len(f)),
            })
            continue

        m_med = float(np.median(m))
        f_med = float(np.median(f))
        p = float(mannwhitneyu(m, f, alternative='two-sided').pvalue)

        observed_dir = 'male > female' if m_med > f_med else ('female > male' if f_med > m_med else 'equal')
        dir_ok = bool(dir_ok_fn(m_med, f_med))
        pass_flag = bool((p <= 0.05) and dir_ok)

        rows.append({
            'Source': source_name,
            'Feature': feat,
            'Male Median': m_med,
            'Female Median': f_med,
            'Abs Diff': abs(m_med - f_med),
            'Expected Direction': expected_dir,
            'Observed Direction': observed_dir,
            'p_value': p,
            'PASS': pass_flag,
            'N_male_beats': int(len(m)),
            'N_female_beats': int(len(f)),
        })

    return pd.DataFrame(rows)


# ------------------------------
# Synthetic male vs female (strict NORM conditions already generated)
# ------------------------------
syn_male_feats = extract_features_from_group(male_ecg, fs=FS)
syn_female_feats = extract_features_from_group(female_ecg, fs=FS)
syn_df = run_tests('Synthetic (NORM)', syn_male_feats, syn_female_feats)


# ------------------------------
# Real male vs female, strict NORM, train split
# ------------------------------
def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out


# Robust fallback for fresh sessions where df_mapped was not created yet
if 'df_mapped' in globals():
    df_real = df_mapped.copy()
else:
    df_real, lbl_itos_dict2, _, _ = load_dataset(target_folder_ptb_xl)

df_real['label_multihot'] = df_real['label_all_numeric'].apply(lambda x: multihot_encode(x, len(label_names)))
df_real['sex_binary'] = df_real['sex'].astype(np.float32)

max_fold_id = int(df_real['strat_fold'].max())
df_train = df_real[df_real['strat_fold'] < max_fold_id - 1].copy()

rng = np.random.default_rng(123)
N_REAL_PER_SEX = 200

# Try strict NORM first: NORM=1 and exactly one active label
is_norm = df_train['label_multihot'].apply(lambda x: x[norm_idx] > 0.5)
is_norm_only = df_train['label_multihot'].apply(lambda x: np.isclose(np.sum(x), 1.0))

male_idx_strict = df_train.index[(df_train['sex_binary'] == 0.0) & is_norm & is_norm_only].to_numpy()
female_idx_strict = df_train.index[(df_train['sex_binary'] == 1.0) & is_norm & is_norm_only].to_numpy()

if len(male_idx_strict) >= N_REAL_PER_SEX and len(female_idx_strict) >= N_REAL_PER_SEX:
    male_idx = male_idx_strict
    female_idx = female_idx_strict
    norm_mode_used = 'strict NORM-only'
else:
    # Fallback: relaxed NORM = NORM present (other labels may also be present)
    male_idx = df_train.index[(df_train['sex_binary'] == 0.0) & is_norm].to_numpy()
    female_idx = df_train.index[(df_train['sex_binary'] == 1.0) & is_norm].to_numpy()
    norm_mode_used = 'relaxed NORM-present'

if len(male_idx) < N_REAL_PER_SEX or len(female_idx) < N_REAL_PER_SEX:
    raise RuntimeError(
        'Not enough train samples per sex even after fallback. '
        f'mode={norm_mode_used}, male={len(male_idx)}, female={len(female_idx)}, need={N_REAL_PER_SEX}'
    )

print(f'Real-data selection mode: {norm_mode_used}')
print(f'Available in mode: male={len(male_idx)}, female={len(female_idx)}')

male_sel = np.sort(rng.choice(male_idx, size=N_REAL_PER_SEX, replace=False))
female_sel = np.sort(rng.choice(female_idx, size=N_REAL_PER_SEX, replace=False))

# Load memmap-backed signals for selected indices
with open(target_folder_ptb_xl / 'df_memmap.pkl', 'rb') as f:
    df_mem = pickle.load(f)

df_mem = df_mem.copy()
np.string_ = np.bytes_


def load_real_signals_by_index(selected_idx):
    sub = df_mem.loc[selected_idx].copy()
    sub['dummy'] = 0.0
    ds = TimeseriesDatasetCrops(
        sub,
        output_size=1000,
        data_folder=target_folder_ptb_xl,
        chunk_length=0,
        min_chunk_length=1000,
        stride=1000,
        transforms=ToTensor(),
        annotation=False,
        col_lbl='dummy',
        memmap_filename=target_folder_ptb_xl / 'memmap.npy',
    )
    xs = []
    for i in range(len(ds)):
        x, _ = ds[i]
        xs.append(x.numpy().astype(np.float32))
    return np.stack(xs)


real_male_ecg = load_real_signals_by_index(male_sel)
real_female_ecg = load_real_signals_by_index(female_sel)

real_male_feats = extract_features_from_group(real_male_ecg, fs=FS)
real_female_feats = extract_features_from_group(real_female_ecg, fs=FS)
real_df = run_tests(f'Real train ({norm_mode_used})', real_male_feats, real_female_feats)


# ------------------------------
# Final summary table
# ------------------------------
summary_df = pd.concat([syn_df, real_df], ignore_index=True)
summary_df = summary_df[
    [
        'Source', 'Feature', 'Male Median', 'Female Median', 'Abs Diff',
        'Expected Direction', 'Observed Direction', 'p_value', 'PASS',
        'N_male_beats', 'N_female_beats'
    ]
]

display(summary_df)

failed = summary_df[~summary_df['PASS']]
if len(failed) > 0:
    print('WARNING: Some tests failed (p>0.05 and/or wrong direction).')
    display(failed)
else:
    print('All tests PASS.')



In [ ]:
# Most common NORM-containing statement combinations in TRAIN split (separate male/female)
import numpy as np
import pandas as pd
from clinical_ts.timeseries_utils import load_dataset

# Robust load (works even in fresh session)
if 'df_mapped' in globals():
    df_combo = df_mapped.copy()
else:
    df_combo, lbl_itos_dict_combo, _, _ = load_dataset(target_folder_ptb_xl)

if 'label_names' not in globals():
    if 'lbl_itos_dict_combo' not in globals():
        _, lbl_itos_dict_combo, _, _ = load_dataset(target_folder_ptb_xl)
    label_names = np.array(lbl_itos_dict_combo['label_all'])

if 'norm_idx' not in globals():
    norm_idx = int(np.where(np.array(label_names) == 'NORM')[0][0])

max_fold_id = int(df_combo['strat_fold'].max())
df_train_combo = df_combo[df_combo['strat_fold'] < max_fold_id - 1].copy()
df_train_combo['sex_binary'] = df_train_combo['sex'].astype(np.float32)


def norm_combo_label(label_idx_list):
    # Keep only rows where NORM is present
    idx = sorted([int(i) for i in label_idx_list])
    if norm_idx not in idx:
        return None
    names = [str(label_names[i]) for i in idx]
    return ' + '.join(names)


def combo_table_for_sex(df_in, sex_val, sex_name, top_k=20):
    sub = df_in[df_in['sex_binary'] == float(sex_val)].copy()
    combo_series = sub['label_all_numeric'].apply(norm_combo_label)
    combo_series = combo_series.dropna()

    if len(combo_series) == 0:
        return pd.DataFrame(columns=['sex', 'norm_combo', 'count', 'percent_of_norm_rows'])

    vc = combo_series.value_counts()
    out = vc.reset_index()
    out.columns = ['norm_combo', 'count']
    out['sex'] = sex_name
    out['percent_of_norm_rows'] = 100.0 * out['count'] / out['count'].sum()
    out = out[['sex', 'norm_combo', 'count', 'percent_of_norm_rows']]
    return out.head(top_k)

male_combo_top = combo_table_for_sex(df_train_combo, sex_val=0.0, sex_name='male', top_k=20)
female_combo_top = combo_table_for_sex(df_train_combo, sex_val=1.0, sex_name='female', top_k=20)

print('Top NORM-containing combinations in TRAIN - male')
display(male_combo_top)
print('Top NORM-containing combinations in TRAIN - female')
display(female_combo_top)

# Optional compact side-by-side view
df_combo_compare = pd.concat([male_combo_top, female_combo_top], ignore_index=True)
display(df_combo_compare)


In [ ]:
# MMD^2 fidelity: real vs synthetic (strict NORM+SR), per sex and per lead
import numpy as np
import pandas as pd
import pickle
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor, load_dataset

# ---------- utilities ----------
def _gaussian_kernel_from_dist2(dist2, sigma):
    return np.exp(-dist2 / (2.0 * (sigma ** 2)))


def _pairwise_dist2(A, B):
    # A: [n,d], B: [m,d]
    A2 = np.sum(A * A, axis=1, keepdims=True)
    B2 = np.sum(B * B, axis=1, keepdims=True).T
    D2 = A2 + B2 - 2.0 * (A @ B.T)
    return np.maximum(D2, 0.0)


def _median_heuristic_sigma(X):
    # X: [n,d], pooled real+synthetic for one sex and one lead
    D2 = _pairwise_dist2(X, X)
    iu = np.triu_indices(D2.shape[0], k=1)
    d = np.sqrt(D2[iu])
    d = d[np.isfinite(d)]
    d = d[d > 0]
    if len(d) == 0:
        return 1.0
    return float(np.median(d))


def _mmd2_unbiased(X, Y, sigma):
    # Unbiased estimator of MMD^2
    m = X.shape[0]
    n = Y.shape[0]
    if m < 2 or n < 2:
        return np.nan

    Kxx = _gaussian_kernel_from_dist2(_pairwise_dist2(X, X), sigma)
    Kyy = _gaussian_kernel_from_dist2(_pairwise_dist2(Y, Y), sigma)
    Kxy = _gaussian_kernel_from_dist2(_pairwise_dist2(X, Y), sigma)

    np.fill_diagonal(Kxx, 0.0)
    np.fill_diagonal(Kyy, 0.0)

    term_xx = Kxx.sum() / (m * (m - 1))
    term_yy = Kyy.sum() / (n * (n - 1))
    term_xy = 2.0 * Kxy.mean()
    return float(term_xx + term_yy - term_xy)


def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[int(i)] = 1.0
    return out


def load_real_signals_by_index(df_mem, selected_idx):
    sub = df_mem.loc[selected_idx].copy()
    sub['dummy'] = 0.0
    ds = TimeseriesDatasetCrops(
        sub,
        output_size=1000,
        data_folder=target_folder_ptb_xl,
        chunk_length=0,
        min_chunk_length=1000,
        stride=1000,
        transforms=ToTensor(),
        annotation=False,
        col_lbl='dummy',
        memmap_filename=target_folder_ptb_xl / 'memmap.npy',
    )
    xs = []
    for i in range(len(ds)):
        x, _ = ds[i]
        xs.append(x.numpy().astype(np.float32))
    return np.stack(xs)


# ---------- inputs ----------
N_MMD = 200
rng = np.random.default_rng(2026)

if 'male_ecg' not in globals() or 'female_ecg' not in globals():
    raise RuntimeError('Synthetic groups not found. Run the NORM+SR generation cell first to create male_ecg and female_ecg.')

if male_ecg.shape[0] < N_MMD or female_ecg.shape[0] < N_MMD:
    raise RuntimeError(
        f'Need at least {N_MMD} synthetic samples per sex. '
        f'Got male={male_ecg.shape[0]}, female={female_ecg.shape[0]}'
    )

syn_male = male_ecg[:N_MMD]
syn_female = female_ecg[:N_MMD]

# ---------- real train strict NORM+SR ----------
if 'df_mapped' in globals():
    df_real = df_mapped.copy()
else:
    df_real, lbl_itos_dict2, _, _ = load_dataset(target_folder_ptb_xl)

if 'label_names' not in globals():
    if 'lbl_itos_dict2' not in globals():
        _, lbl_itos_dict2, _, _ = load_dataset(target_folder_ptb_xl)
    label_names = np.array(lbl_itos_dict2['label_all'])

if 'NORM' not in set(label_names.tolist()) or 'SR' not in set(label_names.tolist()):
    raise ValueError('NORM and/or SR not found in label_names')

norm_idx = int(np.where(label_names == 'NORM')[0][0])
sr_idx = int(np.where(label_names == 'SR')[0][0])

df_real['label_multihot'] = df_real['label_all_numeric'].apply(lambda x: multihot_encode(x, len(label_names)))
df_real['sex_binary'] = df_real['sex'].astype(np.float32)

max_fold_id = int(df_real['strat_fold'].max())
df_train = df_real[df_real['strat_fold'] < max_fold_id - 1].copy()

# strict NORM+SR only => exactly these two active labels
is_norm_sr_strict = df_train['label_multihot'].apply(
    lambda x: (x[norm_idx] > 0.5) and (x[sr_idx] > 0.5) and np.isclose(np.sum(x), 2.0)
)

male_idx = df_train.index[(df_train['sex_binary'] == 0.0) & is_norm_sr_strict].to_numpy()
female_idx = df_train.index[(df_train['sex_binary'] == 1.0) & is_norm_sr_strict].to_numpy()

if len(male_idx) < N_MMD or len(female_idx) < N_MMD:
    raise RuntimeError(
        'Not enough strict NORM+SR train samples per sex for MMD. '
        f'male={len(male_idx)}, female={len(female_idx)}, need={N_MMD}'
    )

real_male_sel = np.sort(rng.choice(male_idx, size=N_MMD, replace=False))
real_female_sel = np.sort(rng.choice(female_idx, size=N_MMD, replace=False))

with open(target_folder_ptb_xl / 'df_memmap.pkl', 'rb') as f:
    df_mem = pickle.load(f)
df_mem = df_mem.copy()
np.string_ = np.bytes_

real_male = load_real_signals_by_index(df_mem, real_male_sel)
real_female = load_real_signals_by_index(df_mem, real_female_sel)

# ---------- lead-wise MMD ----------
rows = []

def compute_sex_rows(sex_name, real_ecg, syn_ecg):
    mmd_vals = []
    sigma_vals = []

    for li, lead_name in enumerate(LEAD_NAMES):
        X = real_ecg[:, li, :].astype(np.float64)   # [n,1000]
        Y = syn_ecg[:, li, :].astype(np.float64)    # [n,1000]

        pooled = np.concatenate([X, Y], axis=0)
        sigma = _median_heuristic_sigma(pooled)  # pooled real+synthetic within sex/lead
        mmd2 = _mmd2_unbiased(X, Y, sigma)

        mmd_vals.append(mmd2)
        sigma_vals.append(sigma)

        rows.append({
            'sex': sex_name,
            'lead': lead_name,
            'n_real': int(X.shape[0]),
            'n_syn': int(Y.shape[0]),
            'sigma': float(sigma),
            'MMD^2': float(mmd2),
        })

    rows.append({
        'sex': sex_name,
        'lead': 'MEAN_ALL_LEADS',
        'n_real': int(real_ecg.shape[0]),
        'n_syn': int(syn_ecg.shape[0]),
        'sigma': float(np.mean(sigma_vals)),
        'MMD^2': float(np.mean(mmd_vals)),
    })


compute_sex_rows('male', real_male, syn_male)
compute_sex_rows('female', real_female, syn_female)

mmd_df = pd.DataFrame(rows)
display(mmd_df)

print('Done. MMD computed with Gaussian kernel and median-heuristic bandwidth (pooled real+synthetic per sex/lead).')